# Telegraf CPU installer preparation

This notebook is intended for CLI execution only. It downloads or copies the vendor Telegraf installer for a selected target OS and architecture. Upload the generated file through **Add Plugin** in MAI. MAI then creates a separate overall plugin installation bundle.

Example:
```bash
TARGET_OS=linux-rpm ARCHITECTURE=x86_64 \
jupyter nbconvert --execute --to notebook --inplace 'jupyter notebooks/cpu-plugin.ipynb'
```

In [ ]:
import hashlib
import json
import os
import shutil
import urllib.parse
import urllib.request
from pathlib import Path

TARGETS = {
    'windows': ('.exe', '.msi', '.zip'),
    'linux-deb': ('.deb',),
    'linux-rpm': ('.rpm',),
    'linux-archive': ('.tar.gz', '.tgz'),
    'macos': ('.pkg', '.dmg', '.tar.gz', '.tgz', '.zip'),
}
target_os = os.environ.get('TARGET_OS', '').strip().lower()
architecture = os.environ.get('ARCHITECTURE', '').strip().lower()
source = os.environ.get('TELEGRAF_SOURCE', '').strip()
telegraf_version = os.environ.get('TELEGRAF_VERSION', '1.40.2').strip()
if target_os not in TARGETS:
    raise ValueError(f'TARGET_OS must be one of: {sorted(TARGETS)}')
if architecture not in {'x86_64', 'arm64'}:
    raise ValueError('ARCHITECTURE must be x86_64 or arm64')
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'backend').is_dir():
    repo_root = repo_root.parent
if not (repo_root / 'backend').is_dir():
    raise RuntimeError('Run this notebook from inside the MAI repository')
if not source:
    package_arch = {
        'windows': {'x86_64': 'amd64', 'arm64': 'arm64'},
        'linux-deb': {'x86_64': 'amd64', 'arm64': 'arm64'},
        'linux-rpm': {'x86_64': 'x86_64', 'arm64': 'aarch64'},
        'linux-archive': {'x86_64': 'amd64', 'arm64': 'arm64'},
        'macos': {'x86_64': 'amd64', 'arm64': 'arm64'},
    }[target_os][architecture]
    package_name = {
        'windows': f'telegraf-{telegraf_version}_windows_{package_arch}.zip',
        'linux-deb': f'telegraf_{telegraf_version}-1_{package_arch}.deb',
        'linux-rpm': f'telegraf-{telegraf_version}-1.{package_arch}.rpm',
        'linux-archive': f'telegraf-{telegraf_version}_linux_{package_arch}.tar.gz',
        'macos': f'telegraf-{telegraf_version}_darwin_{package_arch}.dmg',
    }[target_os]
    source = f'https://dl.influxdata.com/telegraf/releases/{package_name}'
    print(f'Downloading official Telegraf {telegraf_version}: {source}')
output_dir = Path(os.environ.get('OUTPUT_DIR', repo_root / 'generated_plugins')).expanduser().resolve()
output_dir.mkdir(parents=True, exist_ok=True)
source_name = Path(urllib.parse.urlparse(source).path).name
if not source_name or not any(source_name.lower().endswith(ext) for ext in TARGETS[target_os]):
    raise ValueError(f'Telegraf installer must end with one of {TARGETS[target_os]} for {target_os}')
destination = output_dir / source_name
if source.startswith(('https://', 'http://')):
    download_request = urllib.request.Request(source, headers={'User-Agent': 'MAI-Plugin-Packager/1.0'})
    with urllib.request.urlopen(download_request, timeout=120) as response, destination.open('wb') as output:
        shutil.copyfileobj(response, output)
else:
    local_source = Path(source).expanduser()
    if not local_source.is_absolute():
        local_source = repo_root / local_source
    local_source = local_source.resolve()
    if not local_source.is_file():
        raise FileNotFoundError(f'{local_source} does not exist. Remove TELEGRAF_SOURCE to download the official package automatically.')
    if local_source != destination:
        shutil.copy2(local_source, destination)
if destination.stat().st_size == 0:
    raise ValueError('The Telegraf installer is empty')
digest_builder = hashlib.sha256()
with destination.open('rb') as installer_stream:
    for chunk in iter(lambda: installer_stream.read(1024 * 1024), b''):
        digest_builder.update(chunk)
digest = digest_builder.hexdigest()
result = {'target_os': target_os, 'architecture': architecture, 'filename': destination.name, 'path': str(destination), 'size_bytes': destination.stat().st_size, 'checksum_sha256': digest}
print(json.dumps(result, indent=2))
